# Notebook 1 - Coupling and Event Notifications

**Purpose:** experience a synchronous dependency failure and then inspect a durable, human-readable event notification.

An **event** is a fact that happened. An **event notification** is a message that communicates that fact to interested consumers. This notebook calls the small scripts in `lab-code/src` so the command-line lab and notebook tell the same story.

In [ ]:
from pathlib import Path
import json, subprocess, sys
LAB = Path('../lab-code').resolve()
SRC = LAB / 'src'
print('Lab root:', LAB)
subprocess.run([sys.executable, str(SRC / 'reset_outputs.py')], check=True)

## Exercise 1 - Deliberately fail a tight dependency

Predict first: if the notification receiver is unavailable, should this deliberately tightly coupled program write an order? Run the cell and inspect its return code.

In [ ]:
failed = subprocess.run([sys.executable, str(SRC / 'tightly_coupled_order.py'), '--fail-notification'], capture_output=True, text=True)
print(failed.stdout)
print('Return code:', failed.returncode)
print('Committed output exists:', (LAB / 'out/tight/committed_orders.csv').exists())

**Write your observation:** Which secondary dependency became part of the primary success condition? Under what requirement might that be intentional?

In [ ]:
observation_tight = 'Replace with your observation.'
observation_tight

## Exercise 2 - Record a durable notification and recover a consumer

The producer writes a CloudEvents-inspired envelope to a local durable inbox. It does not call the notification receiver directly.

In [ ]:
subprocess.run([sys.executable, str(SRC / 'reset_outputs.py')], check=True)
subprocess.run([sys.executable, str(SRC / 'event_bus_pipeline.py'), 'produce'], check=True)
event = json.loads((LAB / 'out/events/order_notifications.jsonl').read_text().splitlines()[0])
event

In [ ]:
for field in ['id', 'source', 'type', 'time', 'subject', 'data']:
    print(f'{field:12} -> {event[field]}')

In [ ]:
failure = subprocess.run([sys.executable, str(SRC / 'event_bus_pipeline.py'), 'consume', '--fail-notification'], capture_output=True, text=True)
print(failure.stdout)
print('Inbox remains:', (LAB / 'out/events/order_notifications.jsonl').exists())
success = subprocess.run([sys.executable, str(SRC / 'event_bus_pipeline.py'), 'consume'], capture_output=True, text=True, check=True)
print(success.stdout)
print((LAB / 'out/events/notification_checkpoint.json').read_text())

## Checkpoint Questions

1. What changed in the producer's dependency boundary?
2. Why is a checkpoint useful?
3. Why can a retry still require an idempotent receiver?
4. Rewrite the event notification as a single plain-English sentence.